# Training YOLOv8 — Final Project Oprec PENSHIP

Notebook dasar untuk melatih model deteksi `bottle`, `laptop`, dan `pencil` di Google Colab.
Jalankan cell dari atas ke bawah (**Shift + Enter**). Kode ini sengaja dibuat dasar dan **wajib dikembangkan oleh peserta** (lihat bagian Pengembangan kode di guidebook).

## 1. Aktifkan GPU

Menu **Runtime → Change runtime type → T4 GPU → Save**, lalu jalankan cell di bawah.
Apabila muncul tabel berisi nama GPU (misalnya Tesla T4), GPU sudah aktif.

In [ ]:
!nvidia-smi

## 2. Install library

In [ ]:
%pip install -q "ultralytics>=8.3.0" "roboflow>=1.1.0"

## 3. API key Roboflow

Simpan API key di **Colab Secrets**, jangan diketik di notebook (notebook ini ikut dikumpulkan):

1. Ambil API key di https://app.roboflow.com/settings/api
2. Klik ikon kunci (**Secrets**) di panel kiri Colab → **Add new secret**.
3. Name: `ROBOFLOW_API_KEY`, Value: API key Roboflow, lalu aktifkan **Notebook access**.

Apabila secret tidak ditemukan, cell di bawah akan meminta API key lewat kotak input tersembunyi.

In [ ]:
import os

if not os.environ.get("ROBOFLOW_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception:
        from getpass import getpass
        os.environ["ROBOFLOW_API_KEY"] = getpass("Roboflow API key: ")

print("API key is set.")

## 4. Konfigurasi

Workspace, project, dan nomor versi tertera pada URL dataset: `app.roboflow.com/<workspace>/<project>/<version>`,
atau di potongan kode saat **Download Dataset → YOLOv8 → Show download code**.

In [ ]:
# Dataset (Roboflow)
ROBOFLOW_WORKSPACE = "workspace-name"  # from the dataset URL
ROBOFLOW_PROJECT = "project-name"      # from the dataset URL
ROBOFLOW_VERSION = 1                   # dataset version number on Roboflow
DATASET_DIR = "dataset"                # download destination folder

# Training
MODEL = "yolov8n.pt"  # options: yolov8n.pt or yolov8s.pt
EPOCHS = 50
IMGSZ = 640           # must stay 640, do not change
BATCH = 16            # lower to 8 on out-of-memory errors
SEED = 0

## 5. Download dataset

Apabila folder `dataset/` sudah berisi `data.yaml`, download dilewati.

In [ ]:
from pathlib import Path

data_yaml = Path(DATASET_DIR) / "data.yaml"
if data_yaml.exists():
    print(f"Dataset already exists, skipping download: {data_yaml}")
else:
    from roboflow import Roboflow

    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    version = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION)
    dataset = version.download("yolov8", location=DATASET_DIR)
    data_yaml = Path(dataset.location) / "data.yaml"

print(data_yaml.read_text())

## 6. Training

Dengan T4 GPU dan beberapa ratus foto, 50 epoch biasanya selesai dalam hitungan menit sampai sekitar setengah jam.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)  # pretrained weights are downloaded automatically
model.train(data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED)

save_dir = Path(model.trainer.save_dir)
best_path = save_dir / "weights" / "best.pt"
print(f"Training output: {save_dir}")
print(f"Best model     : {best_path}")

## 7. Lihat grafik hasil training

In [ ]:
from IPython.display import Image, display

for name in ["results.png", "confusion_matrix.png"]:
    path = save_dir / name
    if path.exists():
        print(name)
        display(Image(filename=str(path), width=900))

## 8. Evaluasi model terbaik di data validation

In [ ]:
best_model = YOLO(str(best_path))
metrics = best_model.val(data=str(data_yaml), imgsz=IMGSZ)
print(f"mAP50-95: {metrics.box.map:.3f} | mAP50: {metrics.box.map50:.3f}")

## 9. Coba prediksi beberapa foto validation

In [ ]:
valid_images = sorted((Path(DATASET_DIR) / "valid" / "images").glob("*"))[:4]
results = best_model.predict(source=[str(p) for p in valid_images], imgsz=IMGSZ, conf=0.25, save=True)
print(f"Prediction images saved to: {results[0].save_dir}")

for p in sorted(Path(results[0].save_dir).glob("*.jpg")):
    display(Image(filename=str(p), width=500))

## 10. Simpan hasil sebelum sesi Colab berakhir

File di Colab hilang saat sesi ditutup atau terputus. Cell di bawah membuat `runs.zip` (berisi `best.pt`,
`results.csv`, dan grafik) lalu men-download-nya ke laptop.

In [ ]:
import shutil

zip_path = shutil.make_archive("runs", "zip", root_dir=".", base_dir="runs")
print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
    files.download(str(best_path))
except ImportError:
    print("Not running on Colab: copy the files from this folder directly.")